# GroupDNA — WhatsApp Group Chat Analyzer

A Python and NumPy project that analyzes WhatsApp group chat activity, word frequency, response patterns, and participant personality archetypes.

In [1]:
import numpy as np
from datetime import datetime, timedelta

print("GroupDNA setup successful!")
print("NumPy version:", np.__version__)

GroupDNA setup successful!
NumPy version: 2.1.3


In [2]:
# GroupDNA — Select dataset from your computer
# AI-assisted code used for learning.

from google.colab import files
import io

uploaded = files.upload()

filename = next(iter(uploaded))
content = uploaded[filename].decode("utf-8")

with open("/content/hostel_bois.txt", "w", encoding="utf-8") as f:
    f.write(content)

print("Selected file:", filename)
print("Dataset saved as /content/hostel_bois.txt")
print("Total lines:", len(content.splitlines()))

Saving hostel_bois.txt to hostel_bois.txt
Selected file: hostel_bois.txt
Dataset saved as /content/hostel_bois.txt
Total lines: 3178


In [3]:

# GroupDNA — Step 2: WhatsApp Chat Parser
# AI-assisted code used for learning and project development.

from datetime import datetime, timedelta

file_path = "/content/hostel_bois.txt"

messages = []
system_count = 0
media_count = 0
deleted_count = 0

with open(file_path, "r", encoding="utf-8") as file:
    lines = file.readlines()

current_message = None

for line in lines:
    line = line.rstrip("\n\r")

    if not line.strip():
        continue

    # A new WhatsApp message usually contains " - " after its timestamp.
    if " - " in line:
        timestamp_text, message_text = line.split(" - ", 1)

        try:
            timestamp = datetime.strptime(
                timestamp_text.strip(), "%d/%m/%y, %H:%M"
            )
        except ValueError:
            # Treat non-timestamp lines as continuation text.
            if current_message is not None:
                current_message["message"] += "\n" + line
            continue

        # Save the previous message before starting another.
        if current_message is not None:
            text = current_message["message"]

            if "<Media omitted>" in text:
                media_count += 1
            elif "This message was deleted" in text:
                deleted_count += 1
            else:
                messages.append(current_message)

        current_message = None

        # Messages without a sender separator are usually system events.
        if ": " in message_text:
            sender, text = message_text.split(": ", 1)

            current_message = {
                "timestamp": timestamp,
                "sender": sender.strip(),
                "message": text.strip()
            }
        else:
            system_count += 1

    elif current_message is not None:
        # Support messages that continue onto another line.
        current_message["message"] += "\n" + line

# Process the final message in the file.
if current_message is not None:
    text = current_message["message"]

    if "<Media omitted>" in text:
        media_count += 1
    elif "This message was deleted" in text:
        deleted_count += 1
    else:
        messages.append(current_message)

# Identify unique participants.
participants = sorted(set(
    message["sender"] for message in messages
))

# Display parser results.
print("=" * 55)
print("             GROUPDNA CHAT PARSER")
print("=" * 55)
print("Total lines in file :", len(lines))
print("Valid messages      :", len(messages))
print("Participants        :", len(participants))
print("System events       :", system_count)
print("Media omitted       :", media_count)
print("Deleted messages    :", deleted_count)

if messages:
    first_date = min(m["timestamp"] for m in messages)
    last_date = max(m["timestamp"] for m in messages)

    print("First message date  :", first_date)
    print("Last message date   :", last_date)
    print("\nParticipants found:")
    for person in participants:
        print("-", person)

    print("\nFirst 3 parsed messages:")
    for message in messages[:3]:
        print(message)
else:
    print("\nNo messages were parsed. Check the timestamp format.")


             GROUPDNA CHAT PARSER
Total lines in file : 3178
Valid messages      : 3127
Participants        : 6
System events       : 4
Media omitted       : 32
Deleted messages    : 15
First message date  : 2024-04-01 01:17:00
Last message date   : 2024-05-30 23:31:00

Participants found:
- Aman
- Karan
- Neha
- Priya
- Rahul
- Vikas

First 3 parsed messages:
{'timestamp': datetime.datetime(2024, 4, 1, 1, 17), 'sender': 'Rahul', 'message': 'scene fix'}
{'timestamp': datetime.datetime(2024, 4, 1, 1, 17), 'sender': 'Rahul', 'message': 'haan'}
{'timestamp': datetime.datetime(2024, 4, 1, 1, 18), 'sender': 'Rahul', 'message': 'kya scene'}


In [4]:

# GroupDNA — Step 3: Group Overview
# AI-assisted code used for learning and project development.

# Count messages from each participant
message_counts = {}

for message in messages:
    person = message["sender"]

    if person not in message_counts:
        message_counts[person] = 0

    message_counts[person] += 1

total_messages = len(messages)

# Display participant statistics
print("=" * 55)
print("                 GROUP OVERVIEW")
print("=" * 55)
print("Total messages     :", total_messages)
print("Total participants :", len(participants))

if messages:
    first_date = min(m["timestamp"] for m in messages)
    last_date = max(m["timestamp"] for m in messages)
    total_days = (last_date.date() - first_date.date()).days + 1

    print("Date range         :", first_date.date(), "to", last_date.date())
    print("Days covered       :", total_days)

    print("\nMESSAGE COUNT BY PARTICIPANT")
    print("-" * 55)

    sorted_participants = sorted(
        message_counts.items(),
        key=lambda item: item[1],
        reverse=True
    )

    for person, count in sorted_participants:
        percentage = (count / total_messages) * 100
        print(f"{person:<15} {count:>5} messages  ({percentage:>5.1f}%)")

    # Identify the most active participant
    most_active = sorted_participants[0]

    print("\nMOST ACTIVE PARTICIPANT")
    print("-" * 55)
    print("Name            :", most_active[0])
    print("Messages        :", most_active[1])
    print("Share of chat   :", f"{most_active[1] / total_messages * 100:.1f}%")

    # Count messages by calendar day
    daily_counts = {}

    for message in messages:
        day = message["timestamp"].date()

        if day not in daily_counts:
            daily_counts[day] = 0

        daily_counts[day] += 1

    busiest_day = max(daily_counts, key=daily_counts.get)

    print("\nBUSIEST DAY")
    print("-" * 55)
    print("Date            :", busiest_day)
    print("Messages        :", daily_counts[busiest_day])

    # Count messages by hour
    hourly_counts = {}

    for message in messages:
        hour = message["timestamp"].hour

        if hour not in hourly_counts:
            hourly_counts[hour] = 0

        hourly_counts[hour] += 1

    busiest_hour = max(hourly_counts, key=hourly_counts.get)

    print("\nBUSIEST HOUR")
    print("-" * 55)
    print(f"Time            : {busiest_hour:02d}:00–{(busiest_hour + 1) % 24:02d}:00")
    print("Messages        :", hourly_counts[busiest_hour])

print("=" * 55)


                 GROUP OVERVIEW
Total messages     : 3127
Total participants : 6
Date range         : 2024-04-01 to 2024-05-30
Days covered       : 60

MESSAGE COUNT BY PARTICIPANT
-------------------------------------------------------
Rahul             940 messages  ( 30.1%)
Priya             712 messages  ( 22.8%)
Neha              624 messages  ( 20.0%)
Aman              484 messages  ( 15.5%)
Karan             345 messages  ( 11.0%)
Vikas              22 messages  (  0.7%)

MOST ACTIVE PARTICIPANT
-------------------------------------------------------
Name            : Rahul
Messages        : 940
Share of chat   : 30.1%

BUSIEST DAY
-------------------------------------------------------
Date            : 2024-05-04
Messages        : 74

BUSIEST HOUR
-------------------------------------------------------
Time            : 18:00–19:00
Messages        : 244


In [5]:

import numpy as np

activity_matrix = np.zeros((len(participants), 24), dtype=int)


participant_index = {}

for index, person in enumerate(participants):
    participant_index[person] = index


for message in messages:
    person = message["sender"]
    hour = message["timestamp"].hour
    row = participant_index[person]

    activity_matrix[row, hour] += 1


print("=" * 85)
print("                    GROUPDNA — DAILY ACTIVITY HEATMAP")
print("=" * 85)
print("Each row represents a participant; each column represents an hour.")
print()


print(f"{'Participant':<14}", end="")

for hour in range(24):
    print(f"{hour:02d} ", end="")

print()
print("-" * 85)


symbols = []

for row in activity_matrix:
    person_symbols = []
    maximum = np.max(row)

    for count in row:
        if count == 0 or maximum == 0:
            symbol = "."
        else:
            ratio = count / maximum

            if ratio <= 0.25:
                symbol = "."
            elif ratio <= 0.50:
                symbol = "░"
            elif ratio <= 0.75:
                symbol = "▒"
            else:
                symbol = "█"

        person_symbols.append(symbol)

    symbols.append(person_symbols)


for index, person in enumerate(participants):
    print(f"{person:<14}", end="")

    for symbol in symbols[index]:
        print(f"{symbol}  ", end="")

    print()

print("-" * 85)
print("Legend: . = 0–25%   ░ = 25–50%   ▒ = 50–75%   █ = 75–100%")
print("Activity levels are relative to each participant's busiest hour.")
print("=" * 85)


print("\nPEAK ACTIVITY HOUR BY PARTICIPANT")
print("-" * 45)

for index, person in enumerate(participants):
    peak_hour = int(np.argmax(activity_matrix[index]))
    peak_count = int(np.max(activity_matrix[index]))

    print(
        f"{person:<14} {peak_hour:02d}:00–"
        f"{(peak_hour + 1) % 24:02d}:00"
        f" ({peak_count} messages)"
    )


                    GROUPDNA — DAILY ACTIVITY HEATMAP
Each row represents a participant; each column represents an hour.

Participant   00 01 02 03 04 05 06 07 08 09 10 11 12 13 14 15 16 17 18 19 20 21 22 23 
-------------------------------------------------------------------------------------
Aman          ▒  █  █  ▒  █  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  ▒  
Karan         .  .  .  .  .  .  .  .  ░  ░  ▒  ░  █  ▒  █  ▒  ▒  ▒  ▒  █  ▒  ░  .  .  
Neha          .  .  .  .  .  ░  .  .  ▒  █  █  ░  ▒  ▒  ░  .  ▒  █  █  █  ▒  ░  ░  ░  
Priya         .  .  .  .  .  .  .  ░  ▒  █  █  █  █  ▒  ▒  ░  ░  ▒  ▒  █  ▒  ░  ░  .  
Rahul         .  .  .  .  .  .  .  .  .  .  .  .  ▒  ░  ░  ▒  ▒  ░  █  ▒  ░  █  ▒  ▒  
Vikas         .  .  .  .  .  .  .  ░  ▒  ░  ░  .  ░  ▒  .  ░  ░  █  ▒  ▒  ░  ░  ░  ▒  
-------------------------------------------------------------------------------------
Legend: . = 0–25%   ░ = 25–50%   ▒ = 50–75%   █ = 75–100%
Activity levels are relative to each pa

In [6]:

# Common words to ignore
stopwords = {
    "the", "and", "is", "in", "it", "to", "of", "a", "for",
    "on", "i", "you", "that", "this", "are", "was", "be",
    "have", "has", "had", "with", "my", "me", "we", "our",
    "they", "he", "she", "an", "at", "as", "but", "so",
    "if", "or", "not", "do", "dont", "don't", "just", "im",
    "its", "your", "what", "how", "can", "will", "no", "yes",
    "yeah", "ok", "okay", "are", "all", "from", "bro"
}

# Punctuation characters to remove from word beginnings and endings
punctuation = ".,!?;:'\"()[]{}<>-/\\*_~`"

word_counts = {}

for message in messages:
    text = message["message"].lower()


    words = text.split()

    for word in words:
        word = word.strip(punctuation)


        if not word or word in stopwords:
            continue


        if word not in word_counts:
            word_counts[word] = 0

        word_counts[word] += 1


sorted_words = sorted(
    word_counts.items(),
    key=lambda item: item[1],
    reverse=True
)

top_10_words = sorted_words[:10]


print("=" * 55)
print("              TOP 10 MOST-USED WORDS")
print("=" * 55)

if top_10_words:
    highest_count = top_10_words[0][1]

    for rank, (word, count) in enumerate(top_10_words, start=1):
        bar_length = int((count / highest_count) * 30)
        bar = "#" * bar_length

        print(f"{rank:2}. {word:<18} {count:>5}  {bar}")
else:
    print("No words found.")

print("=" * 55)
print("Unique words counted:", len(word_counts))


              TOP 10 MOST-USED WORDS
 1. guys                 318  ##############################
 2. about                274  #########################
 3. hai                  268  #########################
 4. am                   260  ########################
 5. today                257  ########################
 6. his                  217  ####################
 7. which                202  ###################
 8. everyone             187  #################
 9. telling              179  ################
10. up                   172  ################
Unique words counted: 546


In [7]:

# Sort all messages chronologically
chronological_messages = sorted(
    messages,
    key=lambda message: message["timestamp"]
)


reply_times = {}

for person in participants:
    reply_times[person] = []


for index in range(1, len(chronological_messages)):
    previous = chronological_messages[index - 1]
    current = chronological_messages[index]

    if previous["sender"] != current["sender"]:
        gap_minutes = (
            current["timestamp"] - previous["timestamp"]
        ).total_seconds() / 60


        if 0 <= gap_minutes <= 1440:
            reply_times[current["sender"]].append(gap_minutes)

print("=" * 65)
print("                RESPONSE SPEED ANALYSIS")
print("=" * 65)

average_reply_times = {}

for person in participants:
    times = reply_times[person]

    if times:
        average_minutes = sum(times) / len(times)
        average_reply_times[person] = average_minutes

        if average_minutes < 60:
            print(
                f"{person:<15} Average response: "
                f"{average_minutes:.1f} minutes"
            )
        else:
            print(
                f"{person:<15} Average response: "
                f"{average_minutes / 60:.2f} hours"
            )
    else:
        average_reply_times[person] = None
        print(f"{person:<15} Not enough reply data")

print("\nFASTEST AVERAGE RESPONSE")
valid_replies = {
    person: time
    for person, time in average_reply_times.items()
    if time is not None
}

if valid_replies:
    fastest_person = min(valid_replies, key=valid_replies.get)
    print(f"{fastest_person}: {valid_replies[fastest_person]:.1f} minutes")


print("\n" + "=" * 65)
print("                 SILENT STREAK ANALYSIS")
print("=" * 65)

first_day = min(m["timestamp"].date() for m in messages)
last_day = max(m["timestamp"].date() for m in messages)

all_days = [
    first_day + timedelta(days=day)
    for day in range((last_day - first_day).days + 1)
]

longest_silent_streaks = {}

for person in participants:
    active_days = set()

    for message in messages:
        if message["sender"] == person:
            active_days.add(message["timestamp"].date())

    current_streak = 0
    longest_streak = 0
    streak_start = None
    longest_start = None
    longest_end = None

    for day in all_days:
        if day not in active_days:
            if current_streak == 0:
                streak_start = day

            current_streak += 1

            if current_streak > longest_streak:
                longest_streak = current_streak
                longest_start = streak_start
                longest_end = day
        else:
            current_streak = 0
            streak_start = None

    longest_silent_streaks[person] = longest_streak

    if longest_streak > 0:
        print(
            f"{person:<15} {longest_streak:>3} silent days "
            f"({longest_start} to {longest_end})"
        )
    else:
        print(f"{person:<15} No silent days")

print("=" * 65)


                RESPONSE SPEED ANALYSIS
Aman            Average response: 54.9 minutes
Karan           Average response: 36.8 minutes
Neha            Average response: 41.3 minutes
Priya           Average response: 42.6 minutes
Rahul           Average response: 35.2 minutes
Vikas           Average response: 34.9 minutes

FASTEST AVERAGE RESPONSE
Vikas: 34.9 minutes

                 SILENT STREAK ANALYSIS
Aman            No silent days
Karan           No silent days
Neha            No silent days
Priya           No silent days
Rahul           No silent days
Vikas            11 silent days (2024-04-23 to 2024-05-03)


In [8]:

group_mom_words = [
    "okay", "safe", "eat", "sleep", "take care", "are you",
    "please", "reminder", "drink water", "don't forget"
]

comedy_words = ["lol", "lmao", "haha", "rofl", "lmfao"]

archetype_scores = {}
archetype_details = {}

first_day = min(m["timestamp"].date() for m in messages)
last_day = max(m["timestamp"].date() for m in messages)
total_days = (last_day - first_day).days + 1

for person in participants:
    person_messages = [m for m in messages if m["sender"] == person]
    count = len(person_messages)

    if count == 0:
        continue

    word_lengths = [len(m["message"].split()) for m in person_messages]
    average_words = sum(word_lengths) / count

    night_count = sum(
        1 for m in person_messages
        if m["timestamp"].hour >= 23 or m["timestamp"].hour < 5
    )
    night_percentage = night_count / count * 100

    uppercase_count = sum(
        1 for m in person_messages
        if len(m["message"]) >= 3
        and (
            (sum(1 for c in m["message"] if c.isupper()) /
             max(1, sum(1 for c in m["message"] if c.isalpha()))) > 0.7
            or m["message"].count("!") >= 2
        )
    )
    drama_percentage = uppercase_count / count * 100

    question_count = sum(
        1 for m in person_messages
        if m["message"].rstrip().endswith("?")
    )
    question_percentage = question_count / count * 100

    comedy_count = sum(
        1 for m in person_messages
        if any(word in m["message"].lower().split() for word in comedy_words)
    )
    comedy_percentage = comedy_count / count * 100

    group_mom_count = sum(
        1 for m in person_messages
        if any(
            keyword in m["message"].lower()
            for keyword in group_mom_words
        )
    )
    group_mom_percentage = group_mom_count / count * 100

    active_days = set(m["timestamp"].date() for m in person_messages)
    silent_days = total_days - len(active_days)
    silent_percentage = silent_days / total_days * 100

    burst_lengths = []
    current_sender = None
    current_burst = 0

    for m in chronological_messages:
        if m["sender"] == person:
            if current_sender == person:
                current_burst += 1
            else:
                if current_burst > 0:
                    burst_lengths.append(current_burst)
                current_sender = person
                current_burst = 1
        else:
            if current_burst > 0:
                burst_lengths.append(current_burst)
            current_sender = m["sender"]
            current_burst = 0

    if current_burst > 0:
        burst_lengths.append(current_burst)

    average_burst = (
        sum(burst_lengths) / len(burst_lengths)
        if burst_lengths else 0
    )

    scores = {
        "Spammer": average_burst / 3 * 100,
        "Group Mom": group_mom_percentage / 20 * 100,
        "Night Owl": night_percentage / 60 * 100,
        "Storyteller": average_words / 30 * 100,
        "Drama Queen": drama_percentage / 30 * 100,
        "Ghost": silent_percentage / 60 * 100,
        "Comedian": comedy_percentage,
        "Question Master": question_percentage / 25 * 100
    }

    archetype_details[person] = {
        "Messages": count,
        "Average words": round(average_words, 2),
        "Night activity %": round(night_percentage, 2),
        "Drama messages %": round(drama_percentage, 2),
        "Question messages %": round(question_percentage, 2),
        "Comedy messages %": round(comedy_percentage, 2),
        "Group Mom messages %": round(group_mom_percentage, 2),
        "Silent days %": round(silent_percentage, 2),
        "Average burst": round(average_burst, 2)
    }

    archetype_scores[person] = scores

print("=" * 65)
print("                 GROUPDNA PERSONALITY ARCHETYPES")
print("=" * 65)

priority = [
    "Spammer", "Group Mom", "Night Owl", "Storyteller",
    "Drama Queen", "Ghost", "Comedian", "Question Master"
]

for person in participants:
    scores = archetype_scores[person]
    winner = max(priority, key=lambda name: scores[name])

    print(f"\n{person}: {winner}")
    print("  Messages:", archetype_details[person]["Messages"])
    print("  Average words per message:",
          archetype_details[person]["Average words"])
    print("  Night activity:",
          f'{archetype_details[person]["Night activity %"]}%')
    print("  Silent days:",
          f'{archetype_details[person]["Silent days %"]}%')

print("\nTie-breaking priority:", " > ".join(priority))
print("=" * 65)


                 GROUPDNA PERSONALITY ARCHETYPES

Aman: Night Owl
  Messages: 484
  Average words per message: 5.02
  Night activity: 80.37%
  Silent days: 0.0%

Karan: Storyteller
  Messages: 345
  Average words per message: 57.05
  Night activity: 2.03%
  Silent days: 0.0%

Neha: Drama Queen
  Messages: 624
  Average words per message: 5.32
  Night activity: 4.81%
  Silent days: 0.0%

Priya: Group Mom
  Messages: 712
  Average words per message: 5.0
  Night activity: 1.26%
  Silent days: 0.0%

Rahul: Spammer
  Messages: 940
  Average words per message: 2.55
  Night activity: 13.51%
  Silent days: 0.0%

Vikas: Ghost
  Messages: 22
  Average words per message: 1.82
  Night activity: 9.09%
  Silent days: 73.33%

Tie-breaking priority: Spammer > Group Mom > Night Owl > Storyteller > Drama Queen > Ghost > Comedian > Question Master


In [9]:

print("=" * 65)
print("              GROUPDNA — FINAL CHAT REPORT")
print("=" * 65)

print("\n1. GROUP SUMMARY")
print("-" * 65)
print("Total messages       :", len(messages))
print("Total participants   :", len(participants))
print("Date range           :", first_date, "to", last_date)
print("Days analyzed        :", total_days)

print("\n2. PARTICIPANT LEADERBOARD")
print("-" * 65)

for rank, (person, count) in enumerate(sorted_participants, start=1):
    percentage = count / total_messages * 100
    print(f"{rank}. {person:<15} {count:>5} messages ({percentage:.1f}%)")

print("\n3. GROUP ACTIVITY")
print("-" * 65)
print("Busiest day          :", busiest_day)
print("Messages that day    :", daily_counts[busiest_day])
print(
    f"Busiest hour         : {busiest_hour:02d}:00–"
    f"{(busiest_hour + 1) % 24:02d}:00"
)
print("Messages that hour   :", hourly_counts[busiest_hour])

print("\n4. MOST USED WORDS")
print("-" * 65)

for rank, (word, count) in enumerate(top_10_words, start=1):
    print(f"{rank:2}. {word:<20} {count}")

print("\n5. RESPONSE SPEED")
print("-" * 65)

for person in participants:
    avg = average_reply_times.get(person)

    if avg is None:
        print(f"{person:<15} Insufficient reply data")
    elif avg < 60:
        print(f"{person:<15} {avg:.1f} minutes")
    else:
        print(f"{person:<15} {avg / 60:.2f} hours")

print("\n6. LONGEST SILENT STREAKS")
print("-" * 65)

for person in sorted(
    longest_silent_streaks,
    key=lambda name: longest_silent_streaks[name],
    reverse=True
):
    print(f"{person:<15} {longest_silent_streaks[person]} days")

print("\n7. PERSONALITY ARCHETYPES")
print("-" * 65)

for person in participants:
    winner = max(
        archetype_scores[person],
        key=archetype_scores[person].get
    )
    print(f"{person:<15} {winner}")

print("\n8. PARSING SUMMARY")
print("-" * 65)
print("Valid messages       :", len(messages))
print("System events        :", system_count)
print("Media omitted        :", media_count)
print("Deleted messages     :", deleted_count)

print("\n" + "=" * 65)
print("                END OF GROUPDNA REPORT")
print("=" * 65)


              GROUPDNA — FINAL CHAT REPORT

1. GROUP SUMMARY
-----------------------------------------------------------------
Total messages       : 3127
Total participants   : 6
Date range           : 2024-04-01 01:17:00 to 2024-05-30 23:31:00
Days analyzed        : 60

2. PARTICIPANT LEADERBOARD
-----------------------------------------------------------------
1. Rahul             940 messages (30.1%)
2. Priya             712 messages (22.8%)
3. Neha              624 messages (20.0%)
4. Aman              484 messages (15.5%)
5. Karan             345 messages (11.0%)
6. Vikas              22 messages (0.7%)

3. GROUP ACTIVITY
-----------------------------------------------------------------
Busiest day          : 2024-05-04
Messages that day    : 74
Busiest hour         : 18:00–19:00
Messages that hour   : 244

4. MOST USED WORDS
-----------------------------------------------------------------
 1. guys                 318
 2. about                274
 3. hai                  268
 4.


## Project Reflection

Through this project, I learned how to process WhatsApp chat data using Python. I used dictionaries to count messages and words, the datetime module to analyze timestamps, and NumPy to create an hourly activity matrix.

I learned that data cleaning and parsing are important because incorrect input handling can affect the final analysis. In the future, I would improve this project by adding more robust message parsing and interactive visualizations.
